# TactileGen: fine-tune the detector on vertical + horizontal bars (Kaggle)

Runs end to end with **Save & Run All**: download PlotQA, convert and clean it,
fine-tune `models/yolo_vbars/best.pt` on a balanced random subset of the
**train** split, then score on the untouched **test** split.

**Before running:** in the right panel set Accelerator to a GPU (T4 or P100),
turn Internet on, and add the TactileGen code zip as an input (Add Input).

Outputs (model, scores, run settings) are saved to `/kaggle/working/out`.

In [ ]:
# Settings for this run. The subset comes from train/val only; test is never trained on.
PER_TYPE = 10_000      # training charts per type (vertical, horizontal); None = all
VAL_PER_TYPE = 1_000   # validation charts per type, checked during training
EPOCHS = 15
SEED = 0
WORK = "/tmp/tg"                # large scratch disk for the data
OUT = "/kaggle/working/out"     # kept after the run

In [ ]:
import glob, json, os, shutil, subprocess, tarfile, time, zipfile
from pathlib import Path


def run(cmd, cwd=None):
    """Run a shell command, show its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    start = time.time()
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)
    print(f"  done in {time.time() - start:.0f}s", flush=True)


Path(WORK).mkdir(parents=True, exist_ok=True)
Path(OUT).mkdir(parents=True, exist_ok=True)

# Find the uploaded code: the folder that contains src/baseline/train_yolo.py.
hits = glob.glob("/kaggle/input/**/src/baseline/train_yolo.py", recursive=True)
if not hits:
    raise SystemExit("TactileGen code not found under /kaggle/input. Add the code zip as an input.")
CODE = "/tmp/TactileGen"
shutil.rmtree(CODE, ignore_errors=True)
shutil.copytree(Path(hits[0]).parents[2], CODE)
os.chdir(CODE)
print("Code:", CODE)
run("nvidia-smi")
run("df -h /tmp /kaggle/working")

In [ ]:
run('pip install -q "ultralytics>=8.4.190,<8.5" gdown')

## Download PlotQA

Official Google Drive files listed in the PlotQA repository. If a download fails
with a quota or "too many users" message, wait a while and rerun this cell.

In [ ]:
FILES = {  # file IDs from github.com/NiteshMethani/PlotQA (PlotQA_Dataset.md)
    "train_png": "1AYuaPX-Lx7T0GZvnsPgN11Twq2FZbWXL",
    "val_png": "1i74NRCEb-x44xqzAovuglex5d583qeiF",
    "test_png": "1D_WPUy91vOrFl6cJUkE55n3ZuB6Qrc4u",
    "train_ann": "1VzWwxBVrlep17BGZU17GpLuGpwjyWbzq",
    "val_ann": "1CCp1tvMd62LfBrWa6pRdb1lpg68A2yFw",
    "test_ann": "1ikiPqkDgxNilYsU5hbK03T4_x2eDmopP",
}
RAW = Path(WORK) / "plotqa"
RAW.mkdir(exist_ok=True)


def unpack(path, dest):
    """Extract a zip or tar archive into dest and delete it; leave other files alone."""
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            archive.extractall(dest)
    elif tarfile.is_tarfile(path):
        with tarfile.open(path) as archive:
            archive.extractall(dest)
    else:
        return
    path.unlink()


for name, file_id in FILES.items():
    dest = RAW / name
    if dest.exists() and any(dest.rglob("*")):
        print("already downloaded:", name)
        continue
    dest.mkdir(exist_ok=True)
    run(f"gdown --quiet --fuzzy https://drive.google.com/uc?id={file_id} -O {dest}/")
    for downloaded in list(dest.iterdir()):
        if downloaded.is_file():
            unpack(downloaded, dest)
    run(f"du -sh {dest}")

In [ ]:
def find_split(split):
    """The PNG folder and annotations JSON of one split, whatever the archive layout."""
    root = RAW / f"{split}_png"
    folders = [p for p in [root, *root.rglob("*")] if p.is_dir()]
    png_dir = max(folders, key=lambda p: sum(1 for _ in p.glob("*.png")))
    count = sum(1 for _ in png_dir.glob("*.png"))
    if not count:
        raise SystemExit(f"No PNG images found for {split} under {root}")
    anns = list((RAW / f"{split}_ann").rglob("*.json"))
    if len(anns) != 1:
        raise SystemExit(f"Expected one annotations JSON for {split}, found {anns}")
    print(f"{split}: {count} PNGs in {png_dir}; annotations {anns[0]}")
    return png_dir, anns[0]


SPLITS = {split: find_split(split) for split in ("train", "val", "test")}

## Convert to YOLO format, keep vertical + horizontal bar charts, remove leakage

`prepare_plotqa.py` keeps only bar charts and drops broken boxes.
`remove_split_leakage.py` removes images that appear byte-for-byte in more than
one split, keeping the test copy, so no test image is trained on.

In [ ]:
BARS = Path(WORK) / "bars"
CLEAN = Path(WORK) / "bars_clean"
shutil.rmtree(BARS, ignore_errors=True)
shutil.rmtree(CLEAN, ignore_errors=True)
for split, (png_dir, ann) in SPLITS.items():
    run(f"python src/dataset/prepare_plotqa.py --annotations {ann} --images {png_dir} "
        f"--output {BARS} --split {split} --chart-types vbar_categorical hbar_categorical")
run(f"python src/dataset/remove_split_leakage.py --source {BARS} --output {CLEAN}")
shutil.copy(CLEAN / "leakage_report.json", OUT)

## Fine-tune

Starts from the current vertical-bar model and trains on `PER_TYPE` vertical +
`PER_TYPE` horizontal charts. Every setting is saved in `run_config.json`.

In [ ]:
RUN = Path(WORK) / "run"
per_type = "" if PER_TYPE is None else f"--per-type {PER_TYPE}"
val_per_type = "" if VAL_PER_TYPE is None else f"--val-per-type {VAL_PER_TYPE}"
run(f"python -m src.baseline.train_yolo --data-root {CLEAN} --output {RUN} "
    f"--weights models/yolo_vbars/best.pt {per_type} {val_per_type} "
    f"--epochs {EPOCHS} --seed {SEED} --device 0 --workers 4")
NEW = RUN / "train" / "weights" / "best.pt"
shutil.copy(NEW, Path(OUT) / "best_bars_hv.pt")
shutil.copy(RUN / "run_config.json", OUT)
shutil.copy(RUN / "train" / "results.csv", OUT)

## Score on the untouched test split

Horizontal bars with the new model; vertical bars with the new **and** old model,
to check vertical bars did not get worse; mAP for every chart part.

In [ ]:
SCORES = Path(OUT) / "scores"
JOBS = {
    "new_horizontal": f"evaluate --bar-role horizontal_bar --weights {NEW}",
    "new_vertical": f"evaluate --bar-role vertical_bar --weights {NEW}",
    "old_vertical": "evaluate --bar-role vertical_bar --weights models/yolo_vbars/best.pt",
    "new_map": f"validate --weights {NEW}",
}
for name, args in JOBS.items():
    ious = " --match-iou 0.5 --match-iou 0.75 --match-iou 0.9" if args.startswith("evaluate") else ""
    run(f"python -m src.baseline.yolo_bars {args} --data-root {CLEAN} --split test "
        f"--output-dir {SCORES / name} --device 0{ious}")

In [ ]:
def load(name):
    return json.loads((SCORES / name / "metrics.json").read_text())


print("Bar detection on PlotQA test (one-to-one matching, confidence 0.25)")
print(f"{'model / bars':<18}{'IoU':>6}{'P':>8}{'R':>8}{'F1':>8}{'count':>8}{'images':>8}")
for name in ("old_vertical", "new_vertical", "new_horizontal"):
    for r in load(name)["matching_results"]:
        print(f"{name:<18}{r['iou']:>6}{r['precision']:>8.4f}{r['recall']:>8.4f}"
              f"{r['f1']:>8.4f}{r['exact_bar_count_accuracy']:>8.4f}{r['images']:>8}")
m = load("new_map")
print()
print(f"New model on test: mAP50 {m['map50']:.4f}, mAP50-95 {m['map50_95']:.4f}")
for part, value in m["per_class_map50_95"].items():
    print(f"  {part:<16} mAP50-95 {value:.4f}")
print()
print("Saved to", OUT, ":", sorted(os.listdir(OUT)))